# Spectral Images

A long-slit frame is a photograph of a slit, dragged across the detector by wavelength. That single
sentence explains most of what follows: why the frame has the
structure it does, why the two axes mean completely different things, and why almost every
`specreduce` operation is described as acting "along the dispersion axis" or "across the
cross-dispersion axis".

This chapter is about the 2D frame itself: what its axes are, what the detector puts in them,
and how `specreduce` represents a frame in code. It stops short of the one-dimensional `Spectrum`
container that operations *return*; that is [Chapter 3](03-spectrum-objects.ipynb).

In [ ]:
import sys

sys.path.append("../src")

import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits
from astropy.visualization import simple_norm
from specreduce.core import parse_image

from common import DATA_DIR, parse_section, read_data

plt.rc("figure", figsize=(8, 2.4), dpi=110)
plt.rc("font", size=9)

arcs, lamps, obj = read_data(trim=False)  # this chapter is about the frame as recorded
ny, nx = obj.data.shape

print(f"science frame: {obj.data.shape}  {obj.unit}")

## Two axes, two different meanings

The array is 512 × 1024. Those are not two samplings of the same kind of thing.

- The **dispersion axis** (1024 pixels here) runs along wavelength. Moving along it changes the
  color of the light. A cut along this axis is a spectrum.
- The **cross-dispersion axis** (512 pixels) runs along the slit, i.e. across the sky. Moving
  along it changes *where* on the sky you are looking. A cut along this axis is an image of the
  slit, which for a point source is a seeing disc.

The clearest way to see the difference is to take one cut of each through the same frame.

In [ ]:
row, column = 275, 512

fig = plt.figure(figsize=(8, 4.6), constrained_layout=True)
grid = fig.add_gridspec(2, 2, height_ratios=[1.15, 1])

frame = fig.add_subplot(grid[0, :])
frame.imshow(
    obj.data,
    origin="lower",
    aspect="auto",
    cmap=plt.cm.Blues,
    norm=simple_norm(obj.data, stretch="log", vmin=0, vmax=250_000),
)
frame.axhline(row, c="C1", lw=0.8)
frame.axvline(column, c="C3", lw=0.8)
frame.set(xlabel="Dispersion [pix]", ylabel="Cross-dispersion [pix]")

along = fig.add_subplot(grid[1, 0])
along.plot(obj.data[row], lw=0.6, c="C1")
along.set(
    xlim=(0, nx - 1),
    xlabel="Dispersion [pix]",
    ylabel="Signal [e$^-$]",
    title=f"row {row}: a spectrum",
)

across = fig.add_subplot(grid[1, 1])
across.plot(obj.data[:, column], lw=0.6, c="C3")
across.set(
    xlim=(0, ny - 1),
    yscale="log",
    xlabel="Cross-dispersion [pix]",
    title=f"column {column}: an image of the slit",
)

for ax in (along, across):
    ax.title.set_size(9)

The two panels could hardly look less alike. Along the row you get a broad continuum with
absorption dips, recognizably a stellar spectrum. Along the column you get a flat pedestal of sky
with a narrow spike where the star's light falls, plus a smaller spike from a second object that
happened to lie in the slit.

Everything the book does follows from this asymmetry. Tracing (Chapter 4) asks *where the spike
sits* as a function of dispersion. Background subtraction (Chapter 7) measures the pedestal either
side of the spike. Extraction (Chapter 8) collapses the spike into one number per column.
Wavelength calibration (Chapters 9–11) puts physical units on the horizontal axis of the left-hand
panel.

## The axis convention

The current release of `specreduce` expects one orientation: **cross-dispersion on axis 0,
dispersion on axis 1**. In NumPy indexing that means `frame[cross_dispersion, dispersion]`: rows
across the slit, columns along wavelength. This is a restriction of the present code rather than a
design principle, and it is due to be relaxed in a future release; for now, a frame in the other
orientation has to be transposed before `specreduce` sees it.

The FITS header records the orientation of the frame as well:

In [ ]:
header = fits.getheader(DATA_DIR / "osiris_tres_3b.fits.bz2")
print(f"DISPAXIS keyword: {header['DISPAXIS']}")

The `DISPAXIS` header keyword records the same thing in the FITS convention, where axes are
1-based and listed fastest-first, so `DISPAXIS = 1` means the *first* FITS axis (`NAXIS1`, the
1024-pixel one) is the dispersion axis, which is NumPy's **last** axis. The two conventions agree
here; they just count from opposite ends.

If your data arrives the other way round, with dispersion running down the columns, transpose it
before handing it to `specreduce`. The frames in this book were transposed when the dataset was
prepared, precisely so that every chapter can take the convention for granted.

:::{note}
Several `specreduce` classes already accept `disp_axis` and `crossdisp_axis` arguments. They are
the interface through which a configurable orientation will arrive, but in the current release the
code behind them still assumes the layout above, so leave them at their defaults and transpose
instead.
:::

## What the detector actually recorded

A raw frame is not all sky and starlight. Part of it never saw the sky at all, and the FITS header
says which part.

In [ ]:
for keyword in ("NAXIS1", "NAXIS2", "CCDSUM", "GAIN", "DATASEC", "BIASSEC", "TRIMSEC"):
    print(f"  {keyword:8s} = {header[keyword]}")

data_section = parse_section(header["DATASEC"])
bias_section = parse_section(header["BIASSEC"])
print(f"\nDATASEC as numpy slices: rows {data_section[0]}, columns {data_section[1]}")
print(f"BIASSEC as numpy slices: rows {bias_section[0]}, columns {bias_section[1]}")

`BIASSEC` points at the **prescan**: real detector pixels, clocked out and digitized exactly like
the rest, but never exposed to light. Chapter 1 explained how `src/common.py` uses it to measure
the read noise. `DATASEC` marks where the illuminated data begins: row 14 in FITS terms, row 13
in NumPy's 0-based indexing. `TRIMSEC` is different in kind. It does not describe the hardware but
the observatory's *recommendation* of what to keep, the region IRAF's `ccdproc` task would cut
the frame down to, and here it is a conservative one: it starts at row 141 (140 in NumPy), and
it stops at column 1000, dropping the last 24 columns at the red end where the throughput is
falling.

None of the three keywords is part of the FITS standard. They are IRAF conventions, written by
observatories because IRAF's tasks read them, and a frame from another instrument may carry all of
them, some, or none.

`CCDSUM = '4 4'` records the binning: 2 × 2 on-chip when the data were taken, and another 2 × 2
applied when this teaching dataset was made, so each pixel here is the sum of sixteen detector
pixels. `GAIN` converts the digitized counts to electrons.

That leaves the question of where the light actually starts, and the two keywords that speak to it
disagree: `DATASEC` says row 13, `TRIMSEC` says row 140. Neither is a measurement. The *optics* do
not illuminate the slit as far down as `DATASEC` claims, since the light level does not reach its
plateau until around row 65, and the recommended trim is generous by design. The frame itself
has to answer.

In [ ]:
slit_profile = np.median(obj.data, axis=1)
rows = np.arange(ny)

fig, ax = plt.subplots(figsize=(8, 3.0), constrained_layout=True)
ax.plot(rows, slit_profile, lw=0.8)

ax.axvspan(bias_section[0].start, bias_section[0].stop, color="C3", alpha=0.18, lw=0)
ax.axvspan(bias_section[0].stop, 65, color="0.6", alpha=0.18, lw=0)
ax.axvspan(65, ny, color="C2", alpha=0.10, lw=0)
ax.axvline(140, c="0.3", ls=":", lw=0.8)

for x, y, label, ha in (
    (3, 3e4, "prescan\n(BIASSEC)", "left"),
    (39, 4e3, "unilluminated", "center"),
    (144, 4e3, "TRIMSEC", "left"),
    (300, 4e3, "illuminated slit", "center"),
):
    ax.text(x, y, label, size=7, ha=ha, c="0.35")

ax.set(
    yscale="log",
    xlim=(0, ny - 1),
    ylim=(30, 2e5),
    xlabel="Cross-dispersion [pix]",
    ylabel="Median signal [e$^-$]",
)

print(f"prescan median      {np.median(slit_profile[bias_section[0]]):8.1f} e-")
print(f"rows 20-60 median   {np.median(slit_profile[20:60]):8.1f} e-")
print(f"illuminated median  {np.median(slit_profile[100:500]):8.1f} e-")

Two levels, a factor of ten apart. The prescan and the unilluminated strip above it are
indistinguishable at about 60 e⁻, since neither is looking at sky, while the illuminated slit
plateaus near 675 e⁻. The optical edge falls between the two keywords. Rows 13 to about 65 sit *inside*
`DATASEC` and still carry no sky, while `TRIMSEC` does not begin until row 140, some 75 rows after
the light has reached its plateau. `TRIMSEC` is safe, then, but safe by being generous, and the
size of the margin is the observatory's choice rather than a property of the frame.

Anything that samples the frame along the cross-dispersion axis, such as a background window or a
tilt-correction sample row, has to stay inside the illuminated region, and "inside `DATASEC`" is
not a strong enough condition. The simplest way to guarantee it is to cut the frame down once and
never think about the edge again, and that is what the book does. From the next chapter on,
`read_data` trims every frame to `TRIMSEC`. The observatory's recommendation is generous, but
nothing of value goes with it, just 75 rows of sky on a 40-arcsecond slit and two dozen columns
beyond the grism's red cutoff, so every later chapter works on a 372 × 1000 frame with the target
near row 135 instead of row 275. This chapter and the previous one pass `trim=False` because they
are about the detector frame as recorded; nothing else in the book needs to.

The trim lives in `src/common.py` for the same reason the bias subtraction does: it is
detector-level bookkeeping, `ccdproc.trim_image` territory rather than `specreduce`'s, and the
chapters should not have to narrate it.

Notice also that the frames carry **no WCS**:

In [ ]:
world_keywords = [
    k
    for k in header
    if k.startswith(("CTYPE", "CRVAL", "CRPIX", "CD1_", "CD2_", "PC1_", "PC2_"))
]
print(f"world-coordinate keywords in the header: {world_keywords or 'none'}")

That is deliberate. The raw frames inherited a celestial (RA/Dec) solution from the acquisition
image, which is meaningless once the light has been dispersed (one axis is a wavelength, not a
sky coordinate), and leaving it in place would let `CCDData.read` attach a confidently wrong WCS.
The book fits its own wavelength solution instead, in Chapters 10 and 11.

## How `specreduce` reads a frame

Every operation in the package funnels its input through one function,
`specreduce.core.parse_image`. It accepts a `Spectrum`, `CCDData`, `NDData`, `Quantity` or a plain
`ndarray`, and returns a consistently shaped object with units, an uncertainty, a spectral axis and
a mask all guaranteed to be present.

That guarantee is the useful part and also the trap: anything missing gets **invented**.

In [ ]:
from_array = parse_image(np.asarray(obj.data))
from_ccddata = parse_image(obj)

print(f"{'':14}  {'bare ndarray':>22}  {'CCDData from read_data':>24}")
print(f"{'unit':14}  {str(from_array.unit):>22}  {str(from_ccddata.unit):>24}")
print(
    f"{'uncertainty':14}  {type(from_array.uncertainty).__name__:>22}  "
    f"{type(from_ccddata.uncertainty).__name__:>24}"
)
print(
    f"{'variance value':14}  {str(np.unique(from_array.uncertainty.array)):>22}  "
    f"{f'{from_ccddata.uncertainty.array[275, 512]:.0f} at the source':>24}"
)
print(
    f"{'spectral axis':14}  {str(from_array.spectral_axis.unit):>22}  "
    f"{str(from_ccddata.spectral_axis.unit):>24}"
)

:::{warning}
**A bare array gets a variance of exactly 1.0 everywhere.** No error is raised and no warning is
issued. The frame then looks fully specified, and every uncertainty-weighted step downstream
silently does the wrong thing: `HorneExtract` weights every pixel equally, `Background`'s sigma
clipping loses its scale, and `find_arc_lines` sets its detection threshold from a fictional noise
level.

Pass a `CCDData` or `Spectrum` carrying a real uncertainty. That is why `common.read_data` attaches
one and why every chapter loads through it.
:::

The invented unit is `DN`, digitized counts, which is a reasonable guess for a raw frame and
wrong for ours. `read_data` has already multiplied by `GAIN`, so these frames are in electrons, and
`parse_image` picks that up from the `CCDData`. Getting this right matters beyond labeling: photon
statistics apply to detected charge, so an uncertainty computed in DN is not a Poisson uncertainty.

The invented spectral axis is a pixel index, which is honest: before Chapter 10 there is no
wavelength scale to attach.

## Masks

The last thing `parse_image` normalizes is the mask, and it takes a `mask_treatment` argument
controlling how masked and non-finite pixels are handled. Real frames need this: dead pixels,
saturated cores and cosmic rays all have to be excluded from fits without being silently replaced
by nonsense.

Here is the whole option set, exercised on a copy of the science frame with a single bad pixel and
part of a bad column blanked out.

In [ ]:
damaged = np.array(obj.data, dtype=float)
damaged[300, 400] = np.nan  # one dead pixel
damaged[100:110, 700] = np.nan  # part of a bad column

print(f"non-finite pixels introduced: {int(np.isnan(damaged).sum())}\n")
print(
    f"{'mask_treatment':>16}  {'masked':>7}  {'non-finite left':>16}  {'value at [300, 400]':>20}"
)
for treatment in (
    "apply",
    "ignore",
    "propagate",
    "zero_fill",
    "nan_fill",
    "apply_mask_only",
    "apply_nan_only",
):
    parsed = parse_image(damaged, mask_treatment=treatment)
    mask = (
        np.asarray(parsed.mask)
        if parsed.mask is not None
        else np.zeros(damaged.shape, bool)
    )
    values = np.asarray(parsed.data)
    print(
        f"{treatment:>16}  {int(mask.sum()):>7}  {int(np.isnan(values).sum()):>16}  "
        f"{values[300, 400]:>20.1f}"
    )

Reading that table:

- **`apply`** (the default) masks the non-finite pixels and leaves the data alone. This is what you
  want almost always.
- **`ignore`** drops the mask entirely; the NaNs are still in the data, now unflagged. Use it only
  when you know the frame is clean.
- **`propagate`** masks the *entire cross-dispersion column* containing any bad pixel: 11 bad
  pixels become 1024 masked ones, two full columns. Drastic, but appropriate when a fit would be
  biased by a partially sampled column.
- **`zero_fill`** and **`nan_fill`** substitute a fill value and drop the mask. `zero_fill` is
  dangerous in exactly the way it sounds: a zero is a perfectly valid flux value, so the bad pixel
  becomes invisible and biases every sum it enters. `nan_fill` is indistinguishable from `ignore`
  in this demo, because the bad pixels were already NaN, but the two differ when the input carries a mask
  over *finite* pixels: `nan_fill` blanks those to NaN, `ignore` simply forgets them.
- **`apply_mask_only`** honors an existing mask but ignores non-finite values. Note it leaves all
  11 NaNs unmasked here, because the input array carried no mask of its own.
- **`apply_nan_only`** does the reverse: builds a fresh mask from the non-finite values and discards
  any mask that came in.

Not every class supports every option. `FitTrace`, for instance, accepts only `apply`,
`propagate` and `apply_nan_only`, because its fit cannot consume non-finite values at all.

## Summary

- A long-slit frame has two axes with different physical meanings: **dispersion** (wavelength, a
  cut along it is a spectrum) and **cross-dispersion** (position along the slit, a cut along it is
  an image of the slit).
- The current release of `specreduce` expects **cross-dispersion on axis 0, dispersion on
  axis 1**, i.e. `frame[cross_dispersion, dispersion]`. Transpose your data if it arrives
  otherwise. The restriction is slated to be lifted, and the `disp_axis`/`crossdisp_axis` arguments
  are where that will surface, but they are not functional yet.
- The FITS `DISPAXIS` keyword says the same thing in 1-based, fastest-first FITS ordering, so
  `DISPAXIS = 1` corresponds to NumPy's last axis.
- `DATASEC`, `BIASSEC` and `TRIMSEC` mark the data region, the prescan, and the observatory's
  recommended trim. They are IRAF conventions rather than FITS-standard keywords, and none of them
  measures the slit: **the optical edge is elsewhere again**, here around row 65, fifty rows above
  where `DATASEC` starts and seventy-five below where `TRIMSEC` does, and cross-dispersion sampling
  has to respect it.
- From Chapter 3 on, `read_data` **trims every frame to `TRIMSEC`**, so the later chapters work
  on a 372 × 1000 frame with the target near row 135 and never have to steer around the edge.
- These frames deliberately carry **no WCS**: a celestial solution is meaningless on a dispersed
  frame, and the book fits its own wavelength scale.
- `parse_image` normalizes every input into one representation and **fabricates anything missing**.
  A bare `ndarray` silently acquires a **variance of 1.0 everywhere** and a unit of `DN`, which
  quietly breaks every uncertainty-weighted step downstream. Always pass a container with a real
  uncertainty.
- `mask_treatment` controls how bad pixels are handled; `apply` is the sensible default, and
  `zero_fill` deserves particular suspicion.

Next: [Spectrum Objects](03-spectrum-objects.ipynb), the container that `specreduce` operations
consume and return.